# Séance 5 · Git et GitHub : sauvegarder son travail en versions · ⭐

**Niveau : ⭐ Débutant**

Tu connais les **points de sauvegarde** d'un jeu vidéo : tu peux revenir en arrière si tu rates un boss. **Git** fait ça pour ton code et tes notebooks. **GitHub** est le cloud qui garde ces sauvegardes et les montre : c'est ton portfolio, celui que regardent les écoles et les employeurs.

Aujourd'hui, on tape les vraies commandes Git, dans un dossier d'entraînement créé par le notebook. Un deuxième dossier joue le rôle de GitHub : on peut s'entraîner à envoyer et récupérer des sauvegardes sans compte ni réseau. À la fin, on passe au vrai GitHub, depuis Colab.

Ce notebook tourne dans **Google Colab** : rien à installer. `Maj + Entrée` pour exécuter une cellule.

**Livrable de la séance** : ton projet de la séance 2 et tes notebooks des séances 3 et 4 en ligne sur ton dépôt GitHub, avec un README et au moins deux commits.

## Préparation

Git est déjà installé dans Colab. La cellule crée un dossier d'entraînement vide, à l'écart de tes fichiers, et une fonction `git("...")` qui lance une commande Git **dans ce dossier** et affiche ce que Git répond. Dans un terminal, on taperait la même chose précédée de `git` : `git("status")` s'écrit `git status`.

Git répond en anglais. Les mots à repérer sont traduits au fil du notebook : *untracked* = non suivi, *modified* = modifié, *deleted* = supprimé.

In [ ]:
import os
import subprocess
import tempfile

ATELIER = tempfile.mkdtemp(prefix="atelier-git-")      # le dossier d'entraînement, vide
DEPOT = os.path.join(ATELIER, "mon-projet")            # ton futur dépôt
os.makedirs(DEPOT)

def git(commande, dossier=None):
    """Lance « git <commande> » dans le dépôt (ou dans un autre dossier) et affiche la réponse de Git."""
    resultat = subprocess.run("git " + commande, shell=True, cwd=dossier or DEPOT, capture_output=True, text=True)
    print((resultat.stdout + resultat.stderr).strip() or "(Git n'a rien répondu : tout s'est bien passé)")

def ecrire(nom, texte, dossier=None):
    """Écrit un fichier texte dans le dépôt (comme si tu l'avais modifié dans un éditeur)."""
    with open(os.path.join(dossier or DEPOT, nom), "w", encoding="utf-8") as f:
        f.write(texte)

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = booléen, ou fonction sans argument)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as erreur:
        print(f"❌ {nom} : erreur pendant la vérification → {erreur}")
        return False
    print(f"✅ {nom}" if ok else f"❌ {nom} : pas encore, réessaie !")
    return ok

def sortie(commande, dossier=None):
    """Ce que Git répond, en texte (pour les vérifications)."""
    return subprocess.run("git " + commande, shell=True, cwd=dossier or DEPOT, capture_output=True, text=True).stdout

print("Dossier d'entraînement prêt :", DEPOT)

## 1. Pourquoi versionner ?

Sans Git, on finit avec `projet.ipynb`, `projet_v2.ipynb`, `projet_final.ipynb`, `projet_final_VRAI.ipynb`… et personne ne sait ce qui a changé entre deux fichiers. Avec Git, il n'y a qu'**un** fichier, et un **historique** : chaque point de sauvegarde (un **commit**) garde l'état de tous les fichiers, une date, et un message qui dit **pourquoi** on a changé.

| Commande | Dans le jeu vidéo | Ce que ça fait |
|---|---|---|
| `git add fichier` | choisir ce qu'on met dans la sauvegarde | prépare les fichiers modifiés |
| `git commit -m "message"` | créer le point de sauvegarde | enregistre une version, avec un message |
| `git push` | envoyer la sauvegarde sur le cloud | copie tes commits sur GitHub |
| `git pull` | récupérer la sauvegarde sur une autre console | rapatrie ce qui a changé sur GitHub |

## 2. Un dépôt, c'est un dossier qui se souvient

Avant de commencer, Git veut savoir qui signe les sauvegardes. On donne un nom et un e-mail, pour ce dépôt seulement.

In [ ]:
git("init -b main")

Le dossier est devenu un **dépôt** : Git y a créé un dossier caché `.git`, où il rangera tout l'historique.

In [ ]:
print(os.listdir(DEPOT))

Qui signe les commits ? Un nom et un e-mail, rangés dans la configuration du dépôt.

In [ ]:
git('config user.name "Élève de l\'atelier"')
git('config user.email "eleve@atelier.local"')

`git status` est la commande qu'on tape le plus souvent : elle dit où on en est. Pour l'instant, rien à sauvegarder.

In [ ]:
git("status")

## 3. Le premier commit

On crée un premier fichier, comme si tu l'avais écrit dans un éditeur.

In [ ]:
ecrire("notes.md", "# Mes notes de data science\n\n- Séance 2 : pandas, filtrer, trier, regrouper\n")

Git voit le fichier, mais ne le suit pas encore : il est **non suivi** (*untracked*).

In [ ]:
git("status")

`git add` choisit ce qui entrera dans la prochaine sauvegarde.

In [ ]:
git("add notes.md")

Le fichier est maintenant **prêt** à être sauvegardé (*staged*).

In [ ]:
git("status")

`git commit -m` crée le point de sauvegarde, avec un message qui dit pourquoi.

In [ ]:
git('commit -m "Ajoute mes notes de la séance 2"')

`git log` affiche l'historique : un commit, son numéro (le début d'une longue suite de chiffres et de lettres), son auteur, sa date et son message.

In [ ]:
git("log")

## 4. Changer, et voir la différence

On ajoute une ligne aux notes.

In [ ]:
ecrire("notes.md", "# Mes notes de data science\n\n- Séance 2 : pandas, filtrer, trier, regrouper\n- Séance 3 : SQL, SELECT, WHERE, GROUP BY\n")

Git voit que le fichier suivi a changé : il est **modifié** (*modified*).

In [ ]:
git("status")

`git diff` montre exactement ce qui a changé : une ligne ajoutée commence par `+`, une ligne retirée par `-`.

In [ ]:
git("diff")

On choisit le fichier…

In [ ]:
git("add notes.md")

…et on sauvegarde. Le message dit l'intention, pas « modif ».

In [ ]:
git('commit -m "Ajoute les notes de la séance 3 sur SQL"')

`--oneline` résume l'historique : une ligne par commit, le plus récent en haut.

In [ ]:
git("log --oneline")

## 5. Revenir en arrière

Le vrai pouvoir de Git : rien n'est perdu. Simulons un accident : on efface le fichier.

In [ ]:
os.remove(os.path.join(DEPOT, "notes.md"))
print(os.listdir(DEPOT))

Git a remarqué la disparition : le fichier est **supprimé** (*deleted*), mais la dernière sauvegarde le contient encore.

In [ ]:
git("status")

`git restore` remet le fichier tel qu'il était au dernier commit.

In [ ]:
git("restore notes.md")

Il est revenu, avec ses deux lignes.

In [ ]:
print(open(os.path.join(DEPOT, "notes.md"), encoding="utf-8").read())

On peut aussi relire un fichier tel qu'il était dans **n'importe quel** commit : `HEAD` est le dernier, `HEAD~1` celui d'avant.

In [ ]:
git("show HEAD~1:notes.md")

### À toi : un troisième commit

Ajoute une ligne pour la séance 4 dans `notes.md` (avec `ecrire`), puis `add` et `commit`, avec un message qui dit pourquoi. Une commande par ligne, comme dans les cellules au-dessus. Le message ne doit pas être `modif`, `update` ou `test`.

In [ ]:
# À toi

In [ ]:
verifier("Trois commits", len(sortie("log --oneline").splitlines()) >= 3)
verifier("Un vrai message", sortie("log -1 --format=%s").strip().lower() not in ("", "modif", "update", "test", "maj"))
verifier("La séance 4 est dans les notes", "4" in sortie("show HEAD:notes.md"))

<details><summary>Solution</summary>

```python
ecrire("notes.md", "# Mes notes de data science\n\n- Séance 2 : pandas, filtrer, trier, regrouper\n- Séance 3 : SQL, SELECT, WHERE, GROUP BY\n- Séance 4 : nettoyer, doublons, valeurs manquantes\n")
git("add notes.md")
git('commit -m "Ajoute les notes de la séance 4 sur le nettoyage"')
git("log --oneline")
```

</details>

## 6. Ce qu'on ne sauvegarde jamais : `.gitignore`

Certains fichiers ne doivent **jamais** partir sur GitHub : une clé d'API (un mot de passe pour un service), un gros fichier de données. On en crée un pour l'exemple.

In [ ]:
ecrire("cle_api.txt", "sk-ceci-est-un-secret-a-ne-jamais-publier")

Git le voit et propose de le suivre : un `git add .` (« tout ajouter ») l'emporterait avec le reste.

In [ ]:
git("status --short")

Le fichier `.gitignore` liste ce que Git doit faire semblant de ne pas voir.

In [ ]:
ecrire(".gitignore", "cle_api.txt\n*.csv\n")

La clé a disparu de la liste. Seul le `.gitignore` reste, et lui, on le sauvegarde.

In [ ]:
git("status --short")

On sauvegarde la règle, pour qu'elle protège aussi les prochains commits.

In [ ]:
git("add .gitignore")
git('commit -m "Ignore la clé d\'API et les fichiers CSV"')

## 7. Envoyer et récupérer : `push` et `pull`

On crée un deuxième dossier qui joue le rôle de GitHub : un dépôt « nu », qui ne sert qu'à recevoir des sauvegardes.

In [ ]:
GITHUB = os.path.join(ATELIER, "github.git")
git(f"init --bare -b main {GITHUB}", dossier=ATELIER)

On dit à ton dépôt où se trouve son « GitHub » : c'est une **remote**, appelée `origin` par habitude.

In [ ]:
git(f"remote add origin {GITHUB}")

`git push` envoie tous tes commits sur la remote.

In [ ]:
git("push -u origin main")

Maintenant, imagine un **deuxième ordinateur** (celui du CDI, chez toi) : `git clone` copie tout le dépôt, historique compris.

In [ ]:
AUTRE_PC = os.path.join(ATELIER, "autre-ordinateur")
git(f"clone {GITHUB} {AUTRE_PC}", dossier=ATELIER)

Sur cet autre ordinateur, on modifie les notes et on envoie la sauvegarde. Le dépôt cloné a besoin de sa propre signature.

In [ ]:
git('config user.name "Élève de l\'atelier"', dossier=AUTRE_PC)
git('config user.email "eleve@atelier.local"', dossier=AUTRE_PC)

Une nouvelle ligne, écrite sur l'autre ordinateur…

In [ ]:
ancien = open(os.path.join(AUTRE_PC, "notes.md"), encoding="utf-8").read()
ecrire("notes.md", ancien + "- Révisé au CDI\n", dossier=AUTRE_PC)

…sauvegardée et envoyée sur « GitHub ».

In [ ]:
git("commit -am \"Ajoute une révision faite au CDI\"", dossier=AUTRE_PC)
git("push", dossier=AUTRE_PC)

De retour sur ton ordinateur, `git pull` récupère ce qui a changé sur GitHub.

In [ ]:
git("pull")

La ligne écrite au CDI est arrivée, et l'historique est le même sur les deux machines.

In [ ]:
git("log --oneline")

## 8. Le vrai GitHub, depuis Colab

Sur Colab, pas besoin de taper ces commandes pour ton portfolio : le menu fait `add`, `commit` et `push` en un clic. Les étapes précises :

1. Sur https://github.com, ouvre ton dépôt `mon-portfolio-ia` (créé à la séance 0). Si tu ne l'as pas : bouton vert **New** → nom `mon-portfolio-ia` → coche *Add a README file* → **Create repository**.
2. Dans Colab, ouvre ton notebook `02_python_pour_la_data.ipynb` (menu **Fichier → Ouvrir un notebook**, onglet *Google Drive*).
3. Menu **Fichier → Enregistrer une copie dans GitHub**. La première fois, GitHub demande d'autoriser Colab : accepte.
4. Dans la fenêtre : *Dépôt* = `ton-pseudo/mon-portfolio-ia`, *Branche* = `main`, *Chemin du fichier* = `seance-02/02_python_pour_la_data.ipynb`, *Message de commit* = `Ajoute mon analyse Pokémon : 3 questions, 3 graphiques`. Coche *Inclure un lien vers Colab*. Clique **OK**.
5. Retourne sur GitHub, actualise : ton notebook est là, lisible avec ses graphiques. C'est un commit, comme ceux de la section 3.
6. Fais pareil pour tes notebooks des séances 3 et 4 (chemins `seance-03/…` et `seance-04/…`).

Pour rouvrir un notebook depuis GitHub sur un autre ordinateur, c'est l'équivalent d'un `pull` : **Fichier → Ouvrir un notebook → onglet GitHub**.

## 9. Projet : le README de ton portfolio

Un **README** explique ton projet en 30 secondes à quelqu'un qui ne l'a jamais vu. C'est la première chose qu'on lit sur la page d'un dépôt. Complète ta fiche : la cellule d'après génère le texte à coller.

In [ ]:
# À toi : complète ta fiche
mon_projet = {
    "titre": "Analyse des Pokémon avec pandas",
    "auteur": "ton pseudo",
    "dataset": "800 Pokémon et leurs stats (Kaggle : abcsds/pokemon)",
    "question": "Les légendaires sont-ils vraiment plus forts ?",
    "outils": "Python, pandas, matplotlib, Google Colab",
}

La cellule assemble le README au format Markdown : `#` pour le titre, `##` pour les sous-titres.

In [ ]:
readme = f"""# {mon_projet['titre']}

Par {mon_projet['auteur']} · Atelier Python, Data Science et IA générative

## Le dataset
{mon_projet['dataset']}

## Ma question
{mon_projet['question']}

## Outils
{mon_projet['outils']}
"""
print(readme)

Sur la page de ton dépôt, clique sur le crayon ✏️ du `README.md`, colle le texte, puis **Commit changes** avec un message qui dit pourquoi. Ouvre ensuite l'onglet **Commits** : chaque ligne est un point de sauvegarde, comme `git log`. Fais une deuxième modification (ajoute une ligne « À faire ensuite ») et commit : tu as maintenant au moins deux versions, et tu peux revenir à la première.

## À retenir

- **Git** garde un historique de commits ; **GitHub** les garde en ligne et les montre.
- `git status` (où j'en suis), `git add` (choisir), `git commit -m` (sauvegarder avec un message qui dit pourquoi), `git log` (l'historique).
- `git diff` montre ce qui a changé ; `git restore` remet un fichier comme au dernier commit.
- `.gitignore` empêche une clé ou un gros fichier de partir sur GitHub.
- `git push` envoie, `git pull` récupère : le même historique sur toutes tes machines.
- Depuis Colab, **Fichier → Enregistrer une copie dans GitHub** fait `add`, `commit` et `push` en un clic.

## Pour montrer aux autres

1. Montre ton dépôt GitHub : les notebooks, le README, l'onglet **Commits**.
2. Lis ton meilleur message de commit : dit-il **pourquoi** ?
3. Qu'est-ce que tu mettrais dans ton `.gitignore` pour ton propre projet ?

Liens gratuits
- Git expliqué visuellement, en jouant : https://learngitbranching.js.org/?locale=fr_FR
- Le guide GitHub pour débuter : https://docs.github.com/fr/get-started/start-your-journey/hello-world
- L'aide-mémoire Git du cours : `docs/aide-memoire-git.md`